# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RohanRamchandani/FlyRank-ML-Week1/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# No computation needed -- Section 1 is a close reading of the research paper.
print("Section 1 is a methodology reading of the FlyRank research paper (markdown above).")
print("Audit of my own model begins in Section 2.")

Section 1 is a methodology reading of the FlyRank research paper (markdown above).
Audit of my own model begins in Section 2.


### Finding 1: [quote or paraphrase the paper's finding in one line]

**Where does the label come from?** [Is the outcome an independently observed future event, or a
bucket derived from the same window the features come from? If derived, the model may be partly
re-describing its own inputs rather than predicting something new.]

**Does the validation design carry the claim?** [Does the split match the claim's scope? A claim
about generalising to new clients needs a client-grouped holdout; a claim about predicting future
movement needs a time-aware split. If the paper reports a random split but claims forward-looking
performance, that's the gap worth asking about.]

**My question, constructively framed:** [e.g. "Could the authors clarify whether the holdout was
grouped by client? With a shared CMS and editorial cadence across a client's pages, a random split
could let the model learn client-level patterns and then be tested on the same client — which would
make the reported number optimistic relative to the new-client use case the paper describes."]

---

### Finding 2: [second finding]

**Where does the label come from?** [...]

**Does the validation design carry the claim?** [...]

**My question, constructively framed:** [...]

---

**A note on tone:** the paper discloses its methods and was written for a broad audience, so these
are clarifying questions about scope and design, not criticisms of the work. The point of this
exercise is to practise asking them — and then to ask the same ones of my own model below, where
they are harder to ask honestly.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd, numpy as np, json
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

CANDIDATES = ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d",
              "content_age_days", "days_since_last_update", "word_count",
              "avg_position", "ctr", "engagement_rate", "scroll_rate",
              "search_volume", "cpc"]
FEATURES = [c for c in CANDIDATES if c in df.columns]

def prep(frame):
    X = frame[FEATURES].copy()
    if "avg_position" in X.columns:
        X["has_position_data"] = (X["avg_position"] > 0).astype(int)
        X["avg_position"] = X["avg_position"].replace(0, np.nan)
    for c in FEATURES:
        if X[c].isna().any():
            X[f"has_{c}"] = X[c].notna().astype(int)
    return X.replace([np.inf, -np.inf], np.nan).fillna(0)

X_all = prep(df)
y_all = df["is_declining_label"].values

def run(tr, te, label):
    m = RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                               class_weight="balanced_subsample",
                               random_state=42, n_jobs=-1)
    m.fit(X_all.iloc[tr], y_all[tr])
    p = m.predict_proba(X_all.iloc[te])[:, 1]
    yt = y_all[te]
    shared = len(set(df.iloc[tr]["client_id"]) & set(df.iloc[te]["client_id"]))
    return {"split": label,
            "precision_at_50": round(precision_at_k(p, yt, 50), 3),
            "precision_at_20": round(precision_at_k(p, yt, 20), 3),
            "roc_auc": round(roc_auc_score(yt, p), 3),
            "avg_precision": round(average_precision_score(yt, p), 3),
            "clients_in_both_folds": shared}

# BEFORE -- naive random split (clients leak across folds)
tr_r, te_r = train_test_split(np.arange(len(df)), test_size=0.30,
                              random_state=42, stratify=y_all)
# AFTER -- honest client-grouped split
tr_g, te_g = next(GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
                  .split(df, y_all, groups=df["client_id"]))

comparison = pd.DataFrame([run(tr_r, te_r, "BEFORE: random split"),
                           run(tr_g, te_g, "AFTER: client-grouped split")])

print("=== Before / after: same model, same features, same metric ===")
print(comparison.to_string(index=False))

delta = comparison.loc[0, "precision_at_50"] - comparison.loc[1, "precision_at_50"]
print(f"\nPrecision@50 drop when clients stop leaking across folds: {delta:+.3f}")
print("The grouped number is the one I report.")

with open("work/outputs/w06_split_comparison.json", "w") as f:
    json.dump(comparison.to_dict(orient="records"), f, indent=2)
print("Wrote work/outputs/w06_split_comparison.json")

=== Before / after: same model, same features, same metric ===
                      split  precision_at_50  precision_at_20  roc_auc  avg_precision  clients_in_both_folds
       BEFORE: random split             0.98             1.00    0.764          0.777                     31
AFTER: client-grouped split             0.52             0.35    0.621          0.652                      0

Precision@50 drop when clients stop leaking across folds: +0.460
The grouped number is the one I report.
Wrote work/outputs/w06_split_comparison.json


**The improvement being shown: random split → client-grouped split.**

A random split lets pages from the same client land in both train and test. Because a client's
pages share a CMS, template, editorial cadence and topic area, the model can learn
client-specific patterns and then be scored on *the same client's other pages* — which inflates
the number without showing that anything transfers.

The grouped split (`GroupShuffleSplit` on `client_id`) guarantees no client appears on both sides.
That matches the claim I actually want to make: this ranking is useful on a client the model has
never seen.

I expect the grouped number to be **lower**. That drop is not a regression — it's the honest
number, and the random-split number was the misleading one.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import cross_val_score, GroupKFold

BANNED = ["trend_direction", "trend_pct", "is_declining_label", "content_id", "client_id"]

print("--- Check 1 & 2: banned columns in the feature set ---")
found = [c for c in X_all.columns if any(b in c for b in BANNED)]
print("Leaky/ID columns found:", found if found else "NONE -- clean")

print("\n--- Check 3: single-feature AUC (leak signature hunt) ---")
single = []
for c in X_all.columns:
    try:
        single.append((c, round(roc_auc_score(y_all, X_all[c]), 3)))
    except Exception:
        pass
single = pd.DataFrame(single, columns=["feature", "auc_alone"])
single["distance_from_random"] = (single["auc_alone"] - 0.5).abs()
print(single.sort_values("distance_from_random", ascending=False).head(8).to_string(index=False))
print("\nAny single feature at |AUC-0.5| > 0.45 would be a leak flag. Review the table above.")

print("\n--- Check 4: deliberate positive control ---")
X_leak = X_all.copy()
X_leak["LEAK_trend_pct"] = df["trend_pct"].fillna(0)   # the known leak, added on purpose
gkf = GroupKFold(n_splits=3)
clean = cross_val_score(RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
                        X_all, y_all, groups=df["client_id"], cv=gkf, scoring="roc_auc").mean()
leaky = cross_val_score(RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
                        X_leak, y_all, groups=df["client_id"], cv=gkf, scoring="roc_auc").mean()
print(f"Clean feature set  -- grouped CV ROC-AUC: {clean:.3f}")
print(f"With trend_pct     -- grouped CV ROC-AUC: {leaky:.3f}  <- the leak signature")
print("The gap shows what leakage looks like here. trend_pct stays OUT of my real feature set.")

--- Check 1 & 2: banned columns in the feature set ---
Leaky/ID columns found: NONE -- clean

--- Check 3: single-feature AUC (leak signature hunt) ---
          feature  auc_alone  distance_from_random
 content_age_days      0.409                 0.091
  impressions_90d      0.584                 0.084
       word_count      0.565                 0.065
 has_avg_position      0.543                 0.043
has_position_data      0.543                 0.043
   has_word_count      0.540                 0.040
has_search_volume      0.540                 0.040
          has_cpc      0.540                 0.040

Any single feature at |AUC-0.5| > 0.45 would be a leak flag. Review the table above.

--- Check 4: deliberate positive control ---
Clean feature set  -- grouped CV ROC-AUC: 0.652
With trend_pct     -- grouped CV ROC-AUC: 1.000  <- the leak signature
The gap shows what leakage looks like here. trend_pct stays OUT of my real feature set.


Running the Week-3 hunt over my final feature set. Four checks:

1. **Label-derived columns.** `is_declining_label` comes from `trend_direction`, which is computed
   from `trend_pct`. Both must be absent from the feature list — otherwise the model reads the
   answer off a reworded copy of itself.
2. **Identifier columns.** `content_id` and `client_id` are pseudonyms. Useful for grouping and
   splitting, never as features — a model that learns an ID has memorised, not generalised.
3. **Suspiciously strong single features.** Any one feature that alone reaches near-perfect
   separation is a leak signature, not a discovery.
4. **Future-window inputs.** Every feature must be knowable before the decision point. In this
   slice all features are trailing-90-day observations, so there is no forward window to leak —
   but this is also the limitation named below.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Pull the real numbers into the claim so the rewrite isn't hand-waved
best = comparison[comparison["split"].str.startswith("AFTER")].iloc[0]
print("Numbers backing the rewritten claim (honest split only):")
print(f"  Split:            {best['split']}")
print(f"  Precision@50:     {best['precision_at_50']}")
print(f"  Precision@20:     {best['precision_at_20']}")
print(f"  ROC-AUC:          {best['roc_auc']}")
print(f"  Clients in both folds: {best['clients_in_both_folds']}  (0 = honest)")
print(f"  Positive rate in sample: {y_all.mean():.3f}")
print("\nSubstitute [X] in the rewritten claim with the Precision@50 above,")
print("and [Y] with your Week-4 baseline recomputed on the same test fold.")

Numbers backing the rewritten claim (honest split only):
  Split:            AFTER: client-grouped split
  Precision@50:     0.52
  Precision@20:     0.35
  ROC-AUC:          0.621
  Clients in both folds: 0  (0 = honest)
  Positive rate in sample: 0.542

Substitute [X] in the rewritten claim with the Precision@50 above,
and [Y] with your Week-4 baseline recomputed on the same test fold.


**My boldest sentence, as originally written (Week 5):**

> "The random forest beats the hand-written rule roughly 3x on Precision@50, showing that a
> learned model is better at finding pages that need refreshing."

**What's wrong with it.** Three things go further than my evidence:
- *"is better at finding pages that need refreshing"* — my label is `trend_direction == "down"`,
  which is a derived proxy for decline, not a measure of whether a page *needs refreshing*.
  Nothing in my data says a refresh was warranted or would have helped.
- *"beats... roughly 3x"* — that figure came from the starter pipeline on one 30,000-row
  anonymized teaching slice. Stated flatly, it reads like a general benchmark.
- It implies a durable, transferable result from a single split on a single sample.

**Rewritten in safe language:**

> On a 30,000-row anonymized starter slice, under a client-grouped holdout split, the random
> forest's ranking achieved a **measured** Precision@50 of [X] against [Y] for my Week-4
> hand-written rule on the same test fold. This is **directional, decision-support** evidence
> that a learned ranking orders this review queue more usefully than my fixed rule on this data.
> It is **not** evidence that refreshing a flagged page causes recovery, that the result
> generalises beyond this sample, or that it reflects any search engine's ranking behaviour.

**Limitations I'm naming rather than burying:**
- The label is a same-window derived bucket, not an independently observed future outcome. A
  genuinely forward-looking label (prior 90 days → next 30 days) would be the honest upgrade.
- One grouped split with one random seed. Repeated splits or grouped cross-validation would give
  a range rather than a point estimate.
- The starter slice shows a ~54% positive rate, far above what a typical content inventory would
  show — likely a teaching-set construction, so absolute numbers here shouldn't be read as
  real-world rates.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.